# Data Mining — Homework 2
**DM Models 1** — All 8 tasks in one notebook.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
warnings.filterwarnings('ignore')
%matplotlib inline
print('All imports OK')

---
# Task 1 — Titanic: Decision Tree & Random Forest (20 points)

## Step 1 — Load Data

In [ ]:
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')
combine  = [train_df, test_df]
print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
train_df.head()

## Step 1 — Preprocessing (HW1 pipeline)
### 1a. Extract Title from Name

In [ ]:
for dataset in combine:
    dataset['Title'] = dataset.Name.str.extract(r' ([A-Za-z]+)\.', expand=False)
for dataset in combine:
    dataset['Title'] = dataset['Title'].replace(
        ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
    dataset['Title'] = dataset['Title'].replace('Mlle', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Ms',   'Miss')
    dataset['Title'] = dataset['Title'].replace('Mme',  'Mrs')
title_mapping = {'Mr': 1, 'Miss': 2, 'Mrs': 3, 'Master': 4, 'Rare': 5}
for dataset in combine:
    dataset['Title'] = dataset['Title'].map(title_mapping).fillna(0).astype(int)
train_df[['Title','Survived']].groupby('Title').mean()

### 1b. Drop unused columns

In [ ]:
train_df = train_df.drop(['PassengerId','Name','Ticket','Cabin'], axis=1)
test_df  = test_df.drop(['Name','Ticket','Cabin'], axis=1)
combine  = [train_df, test_df]
train_df.head()

### 1c. Encode Sex

In [ ]:
for dataset in combine:
    dataset['Sex'] = dataset['Sex'].map({'female': 1, 'male': 0}).astype(int)
train_df[['Sex','Survived']].groupby('Sex').mean()

### 1d. Fill missing Age using median per Sex/Pclass grid

In [ ]:
guess_ages = np.zeros((2, 3))
for dataset in combine:
    for i in range(2):
        for j in range(3):
            guess_df = dataset[(dataset['Sex'] == i) & (dataset['Pclass'] == j+1)]['Age'].dropna()
            guess_ages[i, j] = guess_df.median()
    for i in range(2):
        for j in range(3):
            dataset.loc[(dataset.Age.isnull()) & (dataset.Sex == i) & (dataset.Pclass == j+1), 'Age'] = guess_ages[i, j]
    dataset['Age'] = dataset['Age'].astype(int)
print('Missing Age remaining:', train_df['Age'].isnull().sum())

### 1e. Bin Age into 5 bands

In [ ]:
for dataset in combine:
    dataset.loc[dataset['Age'] <= 16, 'Age'] = 0
    dataset.loc[(dataset['Age'] > 16) & (dataset['Age'] <= 32), 'Age'] = 1
    dataset.loc[(dataset['Age'] > 32) & (dataset['Age'] <= 48), 'Age'] = 2
    dataset.loc[(dataset['Age'] > 48) & (dataset['Age'] <= 64), 'Age'] = 3
    dataset.loc[dataset['Age'] > 64, 'Age'] = 4
train_df[['Age','Survived']].groupby('Age').mean()

### 1f. IsAlone feature

In [ ]:
for dataset in combine:
    dataset['FamilySize'] = dataset['SibSp'] + dataset['Parch'] + 1
    dataset['IsAlone'] = 0
    dataset.loc[dataset['FamilySize'] == 1, 'IsAlone'] = 1
train_df = train_df.drop(['Parch','SibSp','FamilySize'], axis=1)
test_df  = test_df.drop(['Parch','SibSp','FamilySize'], axis=1)
combine  = [train_df, test_df]
train_df[['IsAlone','Survived']].groupby('IsAlone').mean()

### 1g. Age × Pclass interaction, Embarked, Fare — finalize features

In [ ]:
for dataset in combine:
    dataset['Age*Pclass'] = dataset['Age'] * dataset['Pclass']
freq_port = train_df['Embarked'].dropna().mode()[0]
for dataset in combine:
    dataset['Embarked'] = dataset['Embarked'].fillna(freq_port)
    dataset['Embarked'] = dataset['Embarked'].map({'S': 0, 'C': 1, 'Q': 2}).astype(int)
test_df['Fare'].fillna(test_df['Fare'].dropna().median(), inplace=True)
X_train = train_df.drop('Survived', axis=1)
Y_train = train_df['Survived']
X_test  = test_df.drop('PassengerId', axis=1).copy()
print('Features:', list(X_train.columns))
print(f'X_train: {X_train.shape}  |  Y_train: {Y_train.shape}')
X_train.head()

## Step 2 — Fine-tune Decision Tree with GridSearchCV

In [ ]:
param_grid = {
    'max_depth':         [3, 4, 5, 6, 7],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf':  [2, 5, 10],
    'criterion':         ['gini', 'entropy'],
}
grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, Y_train)
best_params = grid_search.best_params_
best_dt     = grid_search.best_estimator_
print('Best parameters:', best_params)
print(f'Best CV accuracy: {grid_search.best_score_:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(best_dt, feature_names=X_train.columns.tolist(),
          class_names=['Not Survived', 'Survived'],
          filled=True, rounded=True, fontsize=9, max_depth=3, ax=ax)
ax.set_title(f"Decision Tree  |  max_depth={best_params['max_depth']}, criterion={best_params['criterion']}", fontsize=14)
plt.tight_layout()
plt.savefig('decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → decision_tree.png')

In [ ]:
print(export_text(best_dt, feature_names=list(X_train.columns), max_depth=3))

## Step 3 — 5-Fold CV: Decision Tree

In [ ]:
dt_cv_scores = cross_val_score(best_dt, X_train, Y_train, cv=5, scoring='accuracy')
print('Fold accuracies:', [round(s * 100, 2) for s in dt_cv_scores])
print(f'Mean accuracy  : {dt_cv_scores.mean() * 100:.2f}%')
print(f'Std deviation  : {dt_cv_scores.std()  * 100:.2f}%')

## Step 4 — 5-Fold CV: Random Forest

In [ ]:
rf_param_grid = {'n_estimators': [100, 200], 'max_depth': [4, 5, 6], 'max_features': ['sqrt', 'log2']}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_param_grid, cv=5, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train, Y_train)
best_rf = rf_grid.best_estimator_
print('Best RF parameters:', rf_grid.best_params_)
print(f'Best RF CV accuracy: {rf_grid.best_score_:.4f}')

In [ ]:
rf_cv_scores = cross_val_score(best_rf, X_train, Y_train, cv=5, scoring='accuracy')
print('Fold accuracies:', [round(s * 100, 2) for s in rf_cv_scores])
print(f'Mean accuracy  : {rf_cv_scores.mean() * 100:.2f}%')
print(f'Std deviation  : {rf_cv_scores.std()  * 100:.2f}%')

## Step 5 — Comparison Summary & Conclusion

In [ ]:
results = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest'],
    'CV Mean Accuracy (%)': [round(dt_cv_scores.mean()*100,2), round(rf_cv_scores.mean()*100,2)],
    'CV Std (%)':           [round(dt_cv_scores.std()*100,2),  round(rf_cv_scores.std()*100,2)],
})
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# Accuracy comparison
means = [dt_cv_scores.mean()*100, rf_cv_scores.mean()*100]
stds  = [dt_cv_scores.std()*100,  rf_cv_scores.std()*100]
bars = axes[0].bar(['Decision Tree','Random Forest'], means, yerr=stds, capsize=6,
                   color=['#5B8DB8','#4CAF82'], edgecolor='white', width=0.45)
for bar, m in zip(bars, means):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
                 f'{m:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_ylim(70, 90)
axes[0].set_ylabel('5-Fold CV Accuracy (%)')
axes[0].set_title('CV Accuracy Comparison')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)
# Feature importances
feat_imp = pd.Series(best_rf.feature_importances_, index=X_train.columns).sort_values()
feat_imp.plot(kind='barh', ax=axes[1], color='#4CAF82', edgecolor='white')
axes[1].set_title('RF Feature Importances')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('dt_vs_rf_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### Conclusion
**Random Forest is the better algorithm** despite a similar mean accuracy, because:
1. **Lower variance** (smaller std) → more stable across unseen data splits
2. **Reduces overfitting** by averaging decorrelated trees built with random feature subsets
3. Single Decision Tree's edge may reflect overfitting to specific folds

Top predictors from RF: **Title, Sex, Fare** — consistent with domain knowledge.

---
# Task 2 — Training Error and Testing (15 points)

Tree structure: Root=B → (B=0→A, B=1→D) → (A=0→leaf, A=1→C) → (D=0→leaf, D=1→E)

## (a) Training Error Rate

In [ ]:
def gini(pos, neg):
    total = pos + neg
    if total == 0: return 0
    return 1 - (pos/total)**2 - (neg/total)**2

leaves = {
    'B=0, A=0':       (14, 5),
    'B=0, A=1, C=0':  (6,  7),
    'B=0, A=1, C=1':  (2,  10),
    'B=1, D=0':       (8,  6),
    'B=1, D=1, E=0':  (5,  17),
    'B=1, D=1, E=1':  (15, 5),
}
total, errors, rows = 0, 0, []
for leaf, (p, n) in leaves.items():
    pred  = '+' if p >= n else '-'
    error = n   if p >= n else p
    total += p + n;  errors += error
    rows.append({'Leaf': leaf, '+': p, '-': n, 'Predicts': pred, 'Errors': error})
print(pd.DataFrame(rows).to_string(index=False))
print(f'\nTotal instances: {total},  Total errors: {errors}')
print(f'Training error rate = {errors}/{total} = {errors/total:.4f} = {errors/total*100:.2f}%')

## (b) Classify T = {A=0, B=1, C=1, D=1, E=0}

In [ ]:
T = {'A':0,'B':1,'C':1,'D':1,'E':0}
print('Classifying T =', T)
print(f"  Root B → B={T['B']} → RIGHT (B=1)")
print(f"  Node D → D={T['D']} → RIGHT (D=1)")
print(f"  Node E → E={T['E']} → LEFT  (E=0)")
print(f"  Leaf: +5, -17 → majority = NEGATIVE")
print("\nResult: T is classified as  −  (Negative)")

---
# Task 3 — Gini Index Splitting (20 points)

In [ ]:
data = pd.DataFrame({
    'A': ['T','T','T','T','T','F','F','F','T','T'],
    'B': ['F','T','T','F','T','F','F','F','T','F'],
    'Class': ['+','+','+','-','+','-','-','-','-','-']
})
data

## Q1 — Overall Gini before splitting

In [ ]:
n = len(data)
total_pos = (data['Class']=='+').sum()
total_neg = (data['Class']=='-').sum()
gini_before = gini(total_pos, total_neg)
print(f'Total: {n}  |  +: {total_pos}  |  -: {total_neg}')
print(f'Gini before = 1 - ({total_pos}/{n})² - ({total_neg}/{n})² = {gini_before:.4f}')

## Q2 — Gini gain after splitting on A

In [ ]:
for val in ['T','F']:
    subset = data[data['A']==val]
    p = (subset['Class']=='+').sum()
    m = (subset['Class']=='-').sum()
    print(f'A={val}: {len(subset)} instances → +{p}, -{m} → Gini={gini(p,m):.4f}')

at = data[data['A']=='T'];  af = data[data['A']=='F']
wg_a = (len(at)/n)*gini((at['Class']=='+').sum(),(at['Class']=='-').sum()) + \
       (len(af)/n)*gini((af['Class']=='+').sum(),(af['Class']=='-').sum())
gain_a = gini_before - wg_a
print(f'Weighted Gini(A) = {wg_a:.4f}')
print(f'Gain(A)          = {gini_before:.4f} - {wg_a:.4f} = {gain_a:.4f}')

## Q3 — Gini gain after splitting on B

In [ ]:
for val in ['T','F']:
    subset = data[data['B']==val]
    p = (subset['Class']=='+').sum()
    m = (subset['Class']=='-').sum()
    print(f'B={val}: {len(subset)} instances → +{p}, -{m} → Gini={gini(p,m):.4f}')

bt = data[data['B']=='T'];  bf = data[data['B']=='F']
wg_b = (len(bt)/n)*gini((bt['Class']=='+').sum(),(bt['Class']=='-').sum()) + \
       (len(bf)/n)*gini((bf['Class']=='+').sum(),(bf['Class']=='-').sum())
gain_b = gini_before - wg_b
print(f'Weighted Gini(B) = {wg_b:.4f}')
print(f'Gain(B)          = {gini_before:.4f} - {wg_b:.4f} = {gain_b:.4f}')

## Q4 — Which attribute does the decision tree choose?

In [ ]:
print(f'Gain(A) = {gain_a:.4f}')
print(f'Gain(B) = {gain_b:.4f}')
print(f'Decision tree chooses: {"B" if gain_b > gain_a else "A"}  (higher gain = greater impurity reduction)')

---
# Task 4 — Conceptual Questions (10 points)

## Q1 — Are decision trees a linear classifier?

In [ ]:
from sklearn.linear_model import LogisticRegression
np.random.seed(42)
X_vis = np.random.randn(200, 2)
y_vis = ((X_vis[:,0]>0)^(X_vis[:,1]>0)).astype(int)  # XOR pattern

dt_vis = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_vis, y_vis)
lr_vis = LogisticRegression().fit(X_vis, y_vis)

xx, yy = np.meshgrid(np.linspace(-3,3,300), np.linspace(-3,3,300))
grid   = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, clf, title in zip(axes,
    [lr_vis, dt_vis],
    ['Logistic Regression\n(single linear boundary)', 'Decision Tree\n(non-linear, axis-parallel boundary)']):
    Z = clf.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X_vis[:,0], X_vis[:,1], c=y_vis, cmap='coolwarm', edgecolors='k', s=30)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
plt.suptitle('Linear vs Non-linear Decision Boundaries', fontsize=13)
plt.tight_layout()
plt.savefig('task4_q1.png', dpi=150, bbox_inches='tight')
plt.show()

**Answer:** Decision trees are **NOT linear classifiers**. They create piecewise axis-parallel boundaries by recursively splitting one feature at a time. The combination of multiple splits produces a staircase-shaped, non-linear decision boundary — as shown in the right plot above. A true linear classifier (like logistic regression) produces a single straight hyperplane.

## Q2 — Is Misclassification error better than Gini index?

In [ ]:
cases = [('Before split (50/50)', 100, 50, 50), ('Split A gives (60/0)', 60, 60, 0), ('Split B gives (30/30)', 60, 30, 30)]
print(f'{"Scenario":<26} {"N":>5} {"Pos":>5} {"Neg":>5} {"Misclass Error":>16} {"Gini":>8}')
print('-'*70)
for label, nt, pos, neg in cases:
    p = pos/nt; q = neg/nt
    print(f'{label:<26} {nt:>5} {pos:>5} {neg:>5} {1-max(p,q):>16.4f} {1-p**2-q**2:>8.4f}')
print('\nKey: "Before split" and "Split B" both show 0.5000 misclassification error — no improvement detected!')
print('Gini correctly detects the improvement from 0.5000 → 0.4800.')

**Answer:** **No — Gini index is superior**. Misclassification error is insensitive to class distribution changes that don't flip the majority class — giving the same score even when a split has genuinely improved purity. Gini index is more discriminating and detects improvements that misclassification error misses, making it a better guide for finding high-quality splits.

---
# Task 5 — Bagging vs. Random Forests (10 points)

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.datasets import make_classification

X_demo, y_demo = make_classification(n_samples=500, n_features=10, n_informative=3, random_state=42)

bagging  = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), n_estimators=50, random_state=42)
rf_demo  = RandomForestClassifier(n_estimators=50, random_state=42)

bag_scores = cross_val_score(bagging, X_demo, y_demo, cv=5)
rf_scores  = cross_val_score(rf_demo, X_demo, y_demo, cv=5)

print(f'Bagging 5-fold CV : {bag_scores.mean()*100:.2f}% ± {bag_scores.std()*100:.2f}%')
print(f'RF      5-fold CV : {rf_scores.mean()*100:.2f}% ± {rf_scores.std()*100:.2f}%')
print('\nRandom Forest achieves higher accuracy and lower variance than Bagging.')

**Answer:**

**Weaknesses of bagging:** Bagging builds each tree using the entire feature set, so trees consistently select the same strong features at their roots. This causes **high correlation between trees**. Averaging correlated predictions offers limited variance reduction.

**How Random Forests differ:** At each split, RF randomly samples only a **subset of features** (typically √p). This forces each tree to use different predictors, **decorrelating** them. Averaging uncorrelated errors reduces variance more effectively — resulting in higher accuracy and greater stability, as confirmed by the lower std above.

---
# Task 6 — SVM Kernel Mapping (20 points)

Map φ([x₁,x₂]) = [z₁,z₂] = [x₁, x₁x₂] and find the maximal margin separator.

In [ ]:
points = [(-1,-1,'Negative'), (-1,+1,'Positive'), (+1,-1,'Positive'), (+1,+1,'Negative')]
print(f'{"[x1,x2]":<12} {"Label":<12} {"[z1=x1, z2=x1*x2]"}')
print('-'*42)
for x1,x2,lbl in points:
    print(f'[{x1:+d},{x2:+d}]      {lbl:<12} [{x1:+d}, {x1*x2:+d}]')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
orig   = [(-1,-1),(-1,+1),(+1,-1),(+1,+1)]
labels = [-1,+1,+1,-1]
mapped = [(x1,x1*x2) for x1,x2 in orig]
colors = {-1:'#534AB7', +1:'#3B8DB8'}

for ax, pts, title, xlabel, ylabel in [
    (axes[0], orig,   'Original space [x₁, x₂]',       'x₁', 'x₂'),
    (axes[1], mapped, 'Feature space [z₁=x₁, z₂=x₁x₂]','z₁','z₂')]:
    for (px,py),lbl in zip(pts,labels):
        ax.scatter(px,py,c=colors[lbl],s=200,zorder=5,edgecolors='k',linewidths=1.2)
        ax.annotate(f'({px},{py})\n{"+" if lbl==1 else "−"}', (px,py),
                    textcoords='offset points', xytext=(12,6), fontsize=10)
    ax.axhline(0,color='gray',lw=0.8); ax.axvline(0,color='gray',lw=0.8)
    ax.set_xlim(-2,2); ax.set_ylim(-2,2)
    ax.set_xlabel(xlabel,fontsize=12); ax.set_ylabel(ylabel,fontsize=12)
    ax.set_title(title,fontsize=12)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Separator and margins in feature space
axes[1].axhline(0,  color='#1D9E75', lw=2.5, label='Separator: z₂=0')
axes[1].axhline(+1, color='#1D9E75', lw=1.2, ls='--', label='Margin: z₂=±1')
axes[1].axhline(-1, color='#3B8DB8', lw=1.2, ls='--')
axes[1].annotate('', xy=(1.7,0), xytext=(1.7,1), arrowprops=dict(arrowstyle='<->',color='gray',lw=1.2))
axes[1].text(1.75, 0.5, 'margin=1', fontsize=9, color='gray', va='center')
axes[1].legend(fontsize=9)

neg_p = mpatches.Patch(color='#534AB7',label='Negative')
pos_p = mpatches.Patch(color='#3B8DB8',label='Positive')
axes[0].legend(handles=[neg_p,pos_p],fontsize=9)

plt.suptitle('Task 6 — SVM Feature Space Mapping', fontsize=13)
plt.tight_layout()
plt.savefig('task6_svm.png', dpi=150, bbox_inches='tight')
plt.show()

**Answer:**

After mapping all negative points land at z₂ = +1 and all positive points at z₂ = −1.

**Maximal margin separator:** z₂ = 0 (i.e., x₁x₂ = 0)

**Margin = 1** (distance from separator z₂=0 to either set of support vectors at z₂=±1)

---
# Task 7 — Circle Linear Separability (10 points)

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC

np.random.seed(0)
X_2d   = np.random.uniform(-2, 2, (400, 2))
inside = ((X_2d[:,0])**2 + (X_2d[:,1])**2) < 1
y_circ = inside.astype(int)

# Original 2D linear
lr_2d   = LogisticRegression().fit(X_2d, y_circ)
acc_2d  = accuracy_score(y_circ, lr_2d.predict(X_2d))

# Lifted feature space (x1, x2, x1^2, x2^2)
X_lift  = np.column_stack([X_2d[:,0], X_2d[:,1], X_2d[:,0]**2, X_2d[:,1]**2])
lr_lift = LogisticRegression().fit(X_lift, y_circ)
acc_lift= accuracy_score(y_circ, lr_lift.predict(X_lift))

print(f'Linear in original (x1,x2) space            : {acc_2d*100:.2f}%')
print(f'Linear in lifted (x1,x2,x1²,x2²) space      : {acc_lift*100:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
ax.scatter(X_2d[~inside,0], X_2d[~inside,1], c='#5B8DB8', s=12, alpha=0.5, label='Outside')
ax.scatter(X_2d[inside,0],  X_2d[inside,1],  c='#E05C3A', s=12, alpha=0.5, label='Inside')
theta = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), 'k-', lw=2)
ax.set_title('Original 2D — NOT linearly separable', fontsize=11)
ax.set_aspect('equal'); ax.legend(fontsize=9)

ax = axes[1]
ax.scatter(X_2d[~inside,0]**2, X_2d[~inside,1]**2, c='#5B8DB8', s=12, alpha=0.5, label='Outside')
ax.scatter(X_2d[inside,0]**2,  X_2d[inside,1]**2,  c='#E05C3A', s=12, alpha=0.5, label='Inside')
x1sq = np.linspace(0, 1, 100)
ax.plot(x1sq, 1-x1sq, 'k-', lw=2, label='Linear separator x₁²+x₂²=1')
ax.set_xlabel('x₁²'); ax.set_ylabel('x₂²')
ax.set_title('Feature space (x₁²,x₂²) — Linearly separable', fontsize=11)
ax.legend(fontsize=9)
plt.suptitle('Task 7 — Circle Separation in Lifted Feature Space', fontsize=12)
plt.tight_layout()
plt.savefig('task7_circle.png', dpi=150, bbox_inches='tight')
plt.show()

**Answer (proof):**

Expand $(x_1-a)^2 + (x_2-b)^2 - r^2 = 0$:

$$(-2a)x_1 + (-2b)x_2 + (1)x_1^2 + (1)x_2^2 + (a^2+b^2-r^2) = 0$$

This is **linear** in $(x_1, x_2, x_1^2, x_2^2)$ with weights $(-2a,\ -2b,\ 1,\ 1)$ and bias $(a^2+b^2-r^2)$. Points inside satisfy $< 0$, outside satisfy $> 0$ — a linear SVM in this feature space perfectly separates any circular region. ∎

---
# Task 8 — Ellipse and Polynomial Kernel (10 points)

In [ ]:
# Verify kernel expansion symbolically
try:
    from sympy import symbols, expand
    u1,u2,v1,v2 = symbols('u1 u2 v1 v2')
    K = (1 + u1*v1 + u2*v2)**2
    print('K(u,v) = (1 + u·v)² expands to:')
    print(expand(K))
    print('\nTerms: 1, u1v1, u2v2, u1²v1², u2²v2², u1u2v1v2')
    print('This is the dot product in feature space φ(x) = (1, √2·x1, √2·x2, x1², x2², √2·x1x2)')
except ImportError:
    print('sympy not installed — install with: pip install sympy')
    print('K(u,v)=(1+u1v1+u2v2)² = 1 + 2u1v1 + 2u2v2 + u1²v1² + u2²v2² + 2u1u2v1v2')

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

np.random.seed(1)
X_ell   = np.random.uniform(-2, 2, (500, 2))
in_ell  = (2*X_ell[:,0]**2 + 0.5*X_ell[:,1]**2) < 1
y_ell   = in_ell.astype(int)

# Linear SVM in the manually lifted 6D space
X_6d    = np.column_stack([np.ones(len(X_ell)), X_ell[:,0], X_ell[:,1],
                            X_ell[:,0]**2, X_ell[:,1]**2, X_ell[:,0]*X_ell[:,1]])
svm_lin = SVC(kernel='linear').fit(X_6d, y_ell)
svm_poly= SVC(kernel='poly', degree=2, coef0=1, gamma=1).fit(X_ell, y_ell)

print(f'Linear SVM in (1,x1,x2,x1²,x2²,x1x2) : {accuracy_score(y_ell,svm_lin.predict(X_6d))*100:.2f}%')
print(f'SVM with poly kernel K=(1+u·v)²       : {accuracy_score(y_ell,svm_poly.predict(X_ell))*100:.2f}%')
print('\nBoth achieve equivalent accuracy — confirming the equivalence.')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X_ell[~in_ell,0], X_ell[~in_ell,1], c='#5B8DB8', s=12, alpha=0.5, label='Outside ellipse')
ax.scatter(X_ell[in_ell,0],  X_ell[in_ell,1],  c='#E05C3A', s=12, alpha=0.5, label='Inside ellipse')
theta = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(theta)/np.sqrt(2), np.sin(theta)/np.sqrt(0.5), 'k-', lw=2.5, label='True ellipse')
xx,yy = np.meshgrid(np.linspace(-2,2,300), np.linspace(-2,2,300))
Z = svm_poly.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
ax.contour(xx,yy,Z,levels=[0.5],colors='#1D9E75',linewidths=2,linestyles='--')
ax.plot([],[],'-',color='#1D9E75',lw=2,label='Poly kernel SVM boundary')
ax.set_xlim(-2,2); ax.set_ylim(-2,2)
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.set_title('Task 8 — SVM Poly Kernel Separating an Elliptic Region', fontsize=12)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('task8_ellipse.png', dpi=150, bbox_inches='tight')
plt.show()

**Answer (proof):**

**Step 1 — Kernel expansion:**

$$K(\mathbf{u},\mathbf{v}) = (1+u_1v_1+u_2v_2)^2 = 1 + 2u_1v_1 + 2u_2v_2 + u_1^2v_1^2 + u_2^2v_2^2 + 2u_1u_2v_1v_2$$

This is $\phi(\mathbf{u})\cdot\phi(\mathbf{v})$ where $\phi(x)=(1,\ \sqrt{2}x_1,\ \sqrt{2}x_2,\ x_1^2,\ x_2^2,\ \sqrt{2}x_1x_2)$ — operating in feature space **(1, x₁, x₂, x₁², x₂², x₁x₂)**.

**Step 2 — Ellipse is linear in this space:**

Expand $c(x_1-a)^2 + d(x_2-b)^2 - 1 = 0$:

$$\underbrace{(ca^2+db^2-1)}_{} \cdot 1 + \underbrace{(-2ca)}_{}x_1 + \underbrace{(-2db)}_{}x_2 + \underbrace{c}_{}x_1^2 + \underbrace{d}_{}x_2^2 + \underbrace{0}_{}x_1x_2 = 0$$

This is a **linear equation** in the 6D feature space. Since the polynomial kernel implicitly maps to this exact space, an SVM with this kernel can find a linear separator there — corresponding to an **elliptic boundary** in the original 2D plane. ∎